<a href="https://colab.research.google.com/github/aka-Trophy-Hunter/Arabic-OCR-with-Qwen2-VL-LoRA-fine-tuning-/blob/main/Arabic_OCR_with_Qwen2_VL_(LoRA_fine_tuning)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft trl accelerate bitsandbytes qwen-vl-utils datasets jiwer
!git clone https://github.com/mbzuai-oryx/KITAB-Bench.git

Cloning into 'KITAB-Bench'...
remote: Enumerating objects: 614, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 614 (delta 93), reused 101 (delta 41), pack-reused 419 (from 1)
Receiving objects: 100% (614/614), 26.89 MiB | 32.01 MiB/s, done.
Resolving deltas: 100% (246/246), done.


In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [3]:
!ls KITAB-Bench
!ls KITAB-Bench/ocr-eval
!head -60 KITAB-Bench/README.md


charts-eval    layout-eval  lines-eval	pdfs-eval  static	vqa-eval
diagrams-eval  LICENSE	    ocr-eval	README.md  tables-eval
eval.py  metrics.py  models  requirements.txt
# <img src="static/images/kitb.png" width="50" > **KITAB-Bench: A Comprehensive Multi-Domain Benchmark for Arabic OCR and Document Understanding [ACL 2025 🔥]**
![](https://i.imgur.com/waxVImv.png)

[Ahmed Heakl](https://huggingface.co/ahmedheakl) <sup> * </sup> &nbsp;
[Abdullah Sohail](https://github.com/AbdullahSohail15) <sup> * </sup> &nbsp;
[Mukul Ranjan](https://scholar.google.com/citations?user=fFBR0j0AAAAJ&hl=en)<sup> * </sup> &nbsp;
[Rania Hossam](https://scholar.google.com/citations?user=ic1jai8AAAAJ&hl=en)<sup> * </sup> &nbsp;
[Ghazi Shazan Ahmad](https://scholar.google.com/citations?hl=en&user=qxmI8TkAAAAJ) &nbsp;
[Mohamed El-Geish](https://www.linkedin.com/in/elgeish/) &nbsp;
[Omar Maher](https://www.linkedin.com/in/omaher/) &nbsp;
[Zhiqiang Shen](https://zhiqiangshen.com/)&nbsp;
[Fahad Shahbaz Khan](https:

In [4]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(
    model_id, min_pixels=256*28*28, max_pixels=768*28*28  # keeps memory low on T4
)

PROMPT = "Extract all the Arabic text in this image exactly as written. Return only the text."

def ocr(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=512)
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
!cat KITAB-Bench/ocr-eval/eval.py
!ls KITAB-Bench/ocr-eval/models
!grep -n "def " KITAB-Bench/ocr-eval/metrics.py

from argparse import ArgumentParser
import os
from tqdm import tqdm
import json
import datasets
import models
from models import AVAILABLE_MODELS

DEFAULT_PROMPT = "Extract the text in the image. Give me the final text, nothing else."
RESULTS_DIR = "results"
MAX_TOKENS = 500
# Number of samples handed to a batched model at once (see eval_ds_batched).
CHUNK_SIZE = 512
os.makedirs(RESULTS_DIR, exist_ok=True)

ds_ids = [
    "ahmedheakl/arocrbench_patsocr", # answer
    "ahmedheakl/arocrbench_historicalbooks", # answer
    "ahmedheakl/arocrbench_khattparagraph", # answer
    "ahmedheakl/arocrbench_synthesizear",
    "ahmedheakl/arocrbench_historyar",
    "ahmedheakl/arocrbench_adab",
    "ahmedheakl/arocrbench_muharaf",
    "ahmedheakl/arocrbench_onlinekhatt",
    "ahmedheakl/arocrbench_khatt",
    "ahmedheakl/arocrbench_isippt",
    "ahmedheakl/arocrbench_arabicocr",
    "ahmedheakl/arocrbench_hindawi",
    "ahmedheakl/arocrbench_evarest",

]

def get_model(model_name: str, flash_attn: b

In [6]:
!pip install -q arabic-reshaper python-bidi
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

text = "مرحبا بكم في جامعة هريپور"
shaped = get_display(arabic_reshaper.reshape(text))

!wget -q -O font.ttf https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoNaskhArabic/NotoNaskhArabic-Regular.ttf
img = Image.new("RGB", (800, 150), "white")
ImageDraw.Draw(img).text((20, 40), shaped, font=ImageFont.truetype("font.ttf", 48), fill="black")
img.save("test.png")

print("Output:", ocr("test.png"))

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


Output: روبيرة تعاجي في مكب ابحم


In [7]:
import datasets
ds_all = datasets.load_dataset("ahmedheakl/arocrbench_synthesizear", split="train")
ds_ood = datasets.load_dataset("ahmedheakl/arocrbench_hindawi", split="train")
print(ds_all, len(ds_all))
print(ds_ood, len(ds_ood))
print(ds_all[0]["text"][:200])

README.md:   0%|          | 0.00/478 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.38MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/500 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/478 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'text', 'source'],
    num_rows: 500
}) 500
Dataset({
    features: ['image', 'text', 'source'],
    num_rows: 200
}) 200
وَإِذا ما سَأَلَتْنِي عَن مَعْنَى لَفَظَهُ "عرب" عِنْدَ


In [12]:
!pip install -q torchmetrics
import sys
sys.path.append("KITAB-Bench/ocr-eval")
from metrics import preprocess_arabic_text as prep
print(prep("مَرْحَبًا بِكُمْ"))

مرحبا بكم


In [13]:
import sys, json, jiwer
sys.path.append("KITAB-Bench/ocr-eval")
try:
    from metrics import preprocess_arabic_text as prep   # KITAB-Bench's own normalization
except Exception as e:
    print("fallback to no normalization:", e); prep = lambda s: s

ds_all = ds_all.shuffle(seed=42)
n = min(300, len(ds_all) - 30)
train_ds = ds_all.select(range(n))
test_in  = ds_all.select(range(n, n + 30))
test_ood = ds_ood.shuffle(seed=42).select(range(30))

PROMPT = "Extract the text in the image. Give me the final text, nothing else."  # same as KITAB-Bench

def evaluate(ds, name, tag):
    preds, refs = [], []
    for s in ds:
        preds.append(prep(ocr(s["image"])))
        refs.append(prep(s["text"]))
    res = {"CER": jiwer.cer(refs, preds), "WER": jiwer.wer(refs, preds)}
    json.dump({"scores": res, "preds": preds, "refs": refs},
              open(f"{tag}_{name}.json", "w"), ensure_ascii=False, indent=1)
    print(tag, name, res)
    return res

In [14]:
base_in  = evaluate(test_in,  "in_domain", "base")
base_ood = evaluate(test_ood, "ood",       "base")

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


base in_domain {'CER': 0.34938704028021017, 'WER': 0.5893719806763285}
base ood {'CER': 0.8731932864892417, 'WER': 0.9500103284445363}


In [16]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
!pip uninstall -y torchao
!pip install -q transformers peft trl accelerate qwen-vl-utils datasets jiwer arabic-reshaper python-bidi Levenshtein sacrebleu nltk torchmetrics
import sys, json, jiwer, torch, datasets
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
sys.path.append("KITAB-Bench/ocr-eval")
from metrics import preprocess_arabic_text as prep

model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=768*28*28)
PROMPT = "Extract the text in the image. Give me the final text, nothing else."

def ocr(image):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=512)
    return processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

ds_all = datasets.load_dataset("ahmedheakl/arocrbench_synthesizear", split="train").shuffle(seed=42)
ds_ood = datasets.load_dataset("ahmedheakl/arocrbench_hindawi", split="train")
n = min(300, len(ds_all) - 30)
train_ds, test_in = ds_all.select(range(n)), ds_all.select(range(n, n + 30))
test_ood = ds_ood.shuffle(seed=42).select(range(30))
train_small = train_ds.filter(lambda x: len(x["text"]) <= 500)

def evaluate(ds, name, tag):
    preds, refs = [], []
    for s in ds:
        preds.append(prep(ocr(s["image"]))); refs.append(prep(s["text"]))
    res = {"CER": jiwer.cer(refs, preds), "WER": jiwer.wer(refs, preds)}
    json.dump({"scores": res, "preds": preds, "refs": refs}, open(f"{tag}_{name}.json", "w"), ensure_ascii=False, indent=1)
    print(tag, name, res); return res

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

In [3]:
args = TrainingArguments(
    output_dir="out", per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=4,
    fp16=True, logging_steps=5, save_strategy="no", report_to="none",
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_small, data_collator=collate)
trainer.train()

model.save_pretrained("qwen2vl-arabic-ocr-lora")
model.eval(); model.config.use_cache = True

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,0.872819
10,0.753334
15,0.758571
20,0.665430
25,0.609045
30,0.557831
35,0.468518


In [4]:
ft_in  = evaluate(test_in,  "in_domain", "ft")
ft_ood = evaluate(test_ood, "ood",       "ft")

base_in  = {"CER": 0.3494, "WER": 0.5894}
base_ood = {"CER": 0.8732, "WER": 0.9500}

print(f"{'':22}{'CER':>8}{'WER':>8}")
print(f"{'Base  in-domain':22}{base_in['CER']:8.3f}{base_in['WER']:8.3f}")
print(f"{'LoRA  in-domain':22}{ft_in['CER']:8.3f}{ft_in['WER']:8.3f}")
print(f"{'Base  out-of-domain':22}{base_ood['CER']:8.3f}{base_ood['WER']:8.3f}")
print(f"{'LoRA  out-of-domain':22}{ft_ood['CER']:8.3f}{ft_ood['WER']:8.3f}")


ft in_domain {'CER': 0.04313099041533546, 'WER': 0.16740088105726872}
ft ood {'CER': 0.6907561801434449, 'WER': 0.839909109688081}
                           CER     WER
Base  in-domain          0.349   0.589
LoRA  in-domain          0.043   0.167
Base  out-of-domain      0.873   0.950
LoRA  out-of-domain      0.691   0.840


In [6]:
import json
d = json.load(open("ft_in_domain.json"))
errs = sorted(range(30), key=lambda i: -jiwer.cer(d["refs"][i], d["preds"][i]))[:3]
for i in errs:
    print("REF :", d["refs"][i][:150]); print("PRED:", d["preds"][i][:150]); print()

REF : نظاما حراريا يستمد طاقته المحركه من الاشعه
PRED: بظامما خرازيا يشتمم طاقته المحركه مزاله شفه

REF : الذي يجئ بعد سابقيه في هذا السياق، يبدو
PRED: الذي جمي بعد سابقه في فها الساف، يجدو

REF : ايونات الفلور والكلور التي تهاجم جزيئات الاوزون وتحطمها
PRED: ايونات الفللور والكللوريه تهااجم جزيئات الاوزون وتحظمها



In [7]:
!ls
!ls qwen2vl-arabic-ocr-lora

base_in_domain.json  ft_in_domain.json	out			 test.png
base_ood.json	     ft_ood.json	qwen2vl-arabic-ocr-lora
font.ttf	     KITAB-Bench	sample_data
adapter_config.json  adapter_model.safetensors	README.md
